# 02 - Inteligencia de Priorizacao

Objetivo: transformar o ranking causal de fontes em uma regra operacional que escolha os dois melhores telefones por CPF.

Esta etapa entrega dois artefatos pedidos no enunciado: um ranking matematico de confiabilidade das fontes e um algoritmo operacional de escolha. A metrica primaria de escolha e entrega (`delivered` ou `read`); leitura continua monitorada como metrica secundaria de negocio.

A validacao offline abaixo deve ser lida como evidencia historica aproximada, nao como prova final de superioridade. Como ha vies de selecao e baixa cobertura de alguns telefones no holdout, a decisao definitiva fica para o A/B do notebook 03.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from sklearn.preprocessing import StandardScaler

import utils as u

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Imports OK')


## 1. Dados, aparicoes e split temporal


In [ ]:
df_disparo_raw, df_telefone_raw = u.carregar_dados()

df_disparo = u.filtrar_status_invalidos(df_disparo_raw)
df_telefone = u.filtrar_telefones_fixos(df_telefone_raw)
df_disparo['envio_datahora'] = pd.to_datetime(df_disparo['envio_datahora'])
df_disparo = df_disparo.sort_values('envio_datahora').reset_index(drop=True)

df_aparicoes_brutas = u.explodir_aparicoes(df_telefone)
df_aparicoes_fonte = u.preparar_aparicoes_por_fonte(df_aparicoes_brutas)
df_phone_cpf = u.preparar_telefone_cpf(df_aparicoes_brutas)
df_phone_meta = u.preparar_metadados_telefone(df_telefone, df_aparicoes_fonte, df_aparicoes_brutas)

splits = u.criar_splits_temporais(df_disparo, frac_treino=0.60, frac_tuning=0.20)
df_train_disparo = splits['treino']
df_tune_disparo = splits['tuning']
df_test_disparo = splits['teste']
df_dev_disparo = pd.concat([df_train_disparo, df_tune_disparo], ignore_index=True)
cutoff_tuning = splits['cutoff_tuning']
cutoff_teste = splits['cutoff_teste']

print('Periodo observado:', df_disparo['envio_datahora'].min(), '->', df_disparo['envio_datahora'].max())
print('Cutoff tuning:', cutoff_tuning)
print('Cutoff teste:', cutoff_teste)
print('Treino:', len(df_train_disparo))
print('Tuning:', len(df_tune_disparo))
print('Teste:', len(df_test_disparo))


## 2. Ranking causal de sistemas aprendido no treino

O ranking usa somente informacao disponivel antes do envio. Para cada sistema, calculamos:

`taxa_entrega = (read + delivered) / total_disparos`

`score_sistema = min_max(EmpiricalBayesLowerBound(entregas, total_disparos))`

O ranking principal usa limite inferior beta-binomial empirico, que encolhe fontes pequenas para a media global e funciona melhor com atribuicao fracionaria. O Wilson lower bound continua reportado como diagnostico comparavel.


In [ ]:
metricas_sistema_train, df_train_sistema = u.aprender_ranking_sistemas(
    df_train_disparo, df_aparicoes_fonte,
    metodo_atribuicao='full', metodo_ranking='empirical_bayes'
)

rankings_sensibilidade = []
for metodo_atribuicao in ['full', 'fracionario', 'fonte_mais_recente']:
    metricas_tmp = u.calcular_score_sistema(
        u.calcular_metricas_sistema(df_train_sistema, metodo_atribuicao=metodo_atribuicao),
        metodo_ranking='empirical_bayes'
    )
    rankings_sensibilidade.append(metricas_tmp.assign(metodo_atribuicao=metodo_atribuicao))
rankings_sensibilidade = pd.concat(rankings_sensibilidade, ignore_index=True)

display(metricas_sistema_train[[
    'id_sistema', 'total_disparos', 'sucessos_entrega', 'taxa_entrega',
    'taxa_leitura', 'eb_lower_entrega', 'posterior_mean_entrega',
    'wilson_lower_entrega', 'score_sistema'
]])
display(rankings_sensibilidade.pivot_table(index='id_sistema', columns='metodo_atribuicao', values='score_sistema'))

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = metricas_sistema_train.sort_values('score_sistema')
ax.barh(plot_df['id_sistema'].astype(str), plot_df['score_sistema'], color='steelblue', alpha=0.85)
ax.set_xlabel('Score causal da fonte')
ax.set_title('Ranking de fontes aprendido no treino')
plt.tight_layout()
plt.show()


## 3. Base de eventos e grid search do half-life

A recencia entra via decaimento exponencial:

`decaimento = exp(-ln(2) * dias_desde_atualizacao / half_life)`

`score_aparicao = score_sistema * decaimento`

Aparicoes futuras ou sem data nao contribuem para score de origem, quantidade de sistemas causais ou decaimento. Elas recebem `dias_desde_atualizacao = 9999` apenas para manter a penalizacao operacional explicita.

O half-life e escolhido no periodo de tuning. O teste temporal fica isolado para a comparacao offline final, reduzindo inflacao por escolha de hiperparametro.

In [ ]:
df_aparicoes_train_score = u.anexar_score_sistema(df_aparicoes_fonte, metricas_sistema_train)
resultados_grid = []

for half_life in u.HALF_LIFE_GRID:
    df_eventos_hl = u.montar_eventos(df_dev_disparo, df_aparicoes_train_score, df_phone_meta, half_life)
    df_train_modelo, df_tune_modelo, _, prior_ddd_hl, baseline_ddd_hl = u.preparar_matrizes_modelo(df_eventos_hl, cutoff_tuning)

    X_train = df_train_modelo[u.FEATURE_COLS].fillna(0)
    X_tune = df_tune_modelo[u.FEATURE_COLS].fillna(0)
    y_train = df_train_modelo['y_entrega']
    y_tune = df_tune_modelo['y_entrega']

    scaler_hl = StandardScaler()
    X_train_s = scaler_hl.fit_transform(X_train)
    X_tune_s = scaler_hl.transform(X_tune)

    modelo_hl = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=u.SEED)
    modelo_hl.fit(X_train_s, y_train)
    y_proba_tune = modelo_hl.predict_proba(X_tune_s)[:, 1]

    resultados_grid.append({
        'half_life': half_life,
        'auc_entrega_tuning': roc_auc_score(y_tune, y_proba_tune),
        'log_loss_entrega_tuning': log_loss(y_tune, y_proba_tune),
        'brier_entrega_tuning': brier_score_loss(y_tune, y_proba_tune),
    })

df_grid = pd.DataFrame(resultados_grid)
df_grid['rank_auc'] = df_grid['auc_entrega_tuning'].rank(ascending=False)
df_grid['rank_log_loss'] = df_grid['log_loss_entrega_tuning'].rank(ascending=True)
df_grid['rank_brier'] = df_grid['brier_entrega_tuning'].rank(ascending=True)
df_grid['rank_medio'] = df_grid[['rank_auc', 'rank_log_loss', 'rank_brier']].mean(axis=1)
BEST_HALF_LIFE = int(df_grid.loc[df_grid['rank_medio'].idxmin(), 'half_life'])

display(df_grid.sort_values('rank_medio'))
print('Half-life escolhido:', BEST_HALF_LIFE)


## 4. Modelo de validacao temporal


In [ ]:
metricas_sistema_dev, df_dev_sistema = u.aprender_ranking_sistemas(
    df_dev_disparo, df_aparicoes_fonte,
    metodo_atribuicao='full', metodo_ranking='empirical_bayes'
)
df_aparicoes_dev_score = u.anexar_score_sistema(df_aparicoes_fonte, metricas_sistema_dev)
df_eventos_best = u.montar_eventos(df_disparo, df_aparicoes_dev_score, df_phone_meta, BEST_HALF_LIFE)
df_train_modelo, df_val_modelo, _, prior_ddd_train, baseline_ddd_train = u.preparar_matrizes_modelo(df_eventos_best, cutoff_teste)

X_train = df_train_modelo[u.FEATURE_COLS].fillna(0)
X_val = df_val_modelo[u.FEATURE_COLS].fillna(0)
y_train = df_train_modelo['y_entrega']
y_val = df_val_modelo['y_entrega']

scaler_validacao = StandardScaler()
X_train_s = scaler_validacao.fit_transform(X_train)
X_val_s = scaler_validacao.transform(X_val)

modelo_validacao = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=u.SEED)
modelo_validacao.fit(X_train_s, y_train)
y_proba_val = modelo_validacao.predict_proba(X_val_s)[:, 1]
df_val_modelo['score_modelo_validacao'] = y_proba_val

metricas_validacao = {
    'auc_entrega': roc_auc_score(df_val_modelo['y_entrega'], y_proba_val),
    'log_loss_entrega': log_loss(df_val_modelo['y_entrega'], y_proba_val),
    'brier_entrega': brier_score_loss(df_val_modelo['y_entrega'], y_proba_val),
    'taxa_entrega_teste': df_val_modelo['y_entrega'].mean(),
    'taxa_read_teste': df_val_modelo['y_read'].mean(),
    'taxa_read_top_decile': df_val_modelo.sort_values('score_modelo_validacao', ascending=False).head(max(1, int(len(df_val_modelo) * 0.10)))['y_read'].mean(),
}

for chave, valor in metricas_validacao.items():
    print(f'{chave}: {valor:.4f}')

coef_df = pd.DataFrame({'feature': u.FEATURE_COLS, 'coef_padronizado': modelo_validacao.coef_[0]}).sort_values('coef_padronizado', ascending=False)
display(coef_df)


## 5. Algoritmo de escolha e validacao offline

O algoritmo combina origem, atualidade, qualidade do telefone e DDD em um score por telefone. A regra heuristica final e:

`score_heuristico = 0.35 * origem_tempo + 0.15 * DDD + 0.10 * fonte_mais_recente + 0.10 * proprietarios + 0.10 * exclusividade_cpf + 0.08 * qualidade + 0.05 * recencia + 0.04 * sistemas + 0.03 * causalidade`

Os pesos sao intencionalmente arbitrarios (`u.PESOS_HEURISTICA_NOTA`): servem apenas como baseline comparativo para o modelo logistico, nao como afirmacao sobre a importancia relativa dos fatores.

Para cada CPF elegivel, selecionamos os dois telefones de maior score. Os desempates sao: maior score secundario, menor idade do dado e menor identificador de telefone. A simulacao offline compara modelo, heuristica, telefone mais recente, regra alfabetica e random; quando o random supera o candidato, o resultado e marcado como evidencia fraca e segue para validacao por A/B.

In [ ]:
telefones_score_cutoff = u.score_phones_at_reference(
    df_aparicoes_dev_score, df_phone_meta, cutoff_teste, BEST_HALF_LIFE,
    prior_ddd_train, baseline_ddd_train, scaler_validacao, modelo_validacao, u.FEATURE_COLS
)



df_candidatos_cpf = df_phone_cpf.merge(telefones_score_cutoff, on='telefone_numero', how='inner')
cpfs_com_2 = df_candidatos_cpf.groupby('cpf')['telefone_numero'].nunique().loc[lambda s: s >= 2].index
df_candidatos_cpf = df_candidatos_cpf[df_candidatos_cpf['cpf'].isin(cpfs_com_2)].copy()

holdout_por_telefone = df_val_modelo.groupby('contato_telefone').agg(
    total_validacao=('id_disparo', 'nunique'),
    taxa_entrega_validacao=('y_entrega', 'mean'),
    taxa_read_validacao=('y_read', 'mean'),
).reset_index().rename(columns={'contato_telefone': 'telefone_numero'})

selecoes = u.gerar_selecoes(df_candidatos_cpf, incluir_random=True)
resumo_metodos, selecoes_avaliadas, metricas_top2_cpf_raw, metricas_top2_cpf_detalhe = u.avaliar_selecao(selecoes, holdout_por_telefone)

resumo_random = resumo_metodos[resumo_metodos['metodo'].str.startswith('random_')].mean(numeric_only=True).to_frame().T
resumo_random['metodo'] = 'random_media_20_seeds'
resumo_deterministico = resumo_metodos[~resumo_metodos['metodo'].str.startswith('random_')].copy()
resumo_comparacao = pd.concat([resumo_deterministico, resumo_random], ignore_index=True)
resumo_comparacao = resumo_comparacao.sort_values(['taxa_entrega_top1', 'taxa_read_top1'], ascending=False)

metricas_top2_random = metricas_top2_cpf_raw[metricas_top2_cpf_raw['metodo'].str.startswith('random_')].mean(numeric_only=True).to_frame().T
metricas_top2_random['metodo'] = 'random_media_20_seeds'
metricas_top2_deterministico = metricas_top2_cpf_raw[~metricas_top2_cpf_raw['metodo'].str.startswith('random_')].copy()
metricas_top2_cpf = pd.concat([metricas_top2_deterministico, metricas_top2_random], ignore_index=True)
metricas_top2_cpf = metricas_top2_cpf.sort_values(['prob_ao_menos_uma_entrega_proxy', 'prob_ao_menos_um_read_proxy'], ascending=False)

display(resumo_comparacao)
display(metricas_top2_cpf)

baselines_deterministicos = ['heuristica', 'mais_recente', 'fonte_mais_recente', 'alfabetico']
modelo_entrega = resumo_deterministico.loc[resumo_deterministico['metodo'] == 'modelo', 'taxa_entrega_top1'].iloc[0]
melhor_baseline = resumo_deterministico[resumo_deterministico['metodo'].isin(baselines_deterministicos)].sort_values(
    ['taxa_entrega_top1', 'taxa_read_top1'], ascending=False
).iloc[0]

if modelo_entrega > melhor_baseline['taxa_entrega_top1']:
    CHAMPION_METODO = 'modelo'
    CHAMPION_MOTIVO = 'Modelo superou os baselines deterministicos em entrega top1 no holdout.'
else:
    CHAMPION_METODO = melhor_baseline['metodo']
    CHAMPION_MOTIVO = 'Modelo nao superou os baselines deterministicos; escolhido melhor metodo deterministico nao aleatorio.'

random_top1 = resumo_comparacao.loc[resumo_comparacao['metodo'] == 'random_media_20_seeds', 'taxa_entrega_top1'].iloc[0]
champion_top1 = resumo_comparacao.loc[resumo_comparacao['metodo'] == CHAMPION_METODO, 'taxa_entrega_top1'].iloc[0]
cobertura_champion = resumo_comparacao.loc[resumo_comparacao['metodo'] == CHAMPION_METODO, 'cobertura_holdout'].iloc[0]
random_superou_champion = bool(random_top1 > champion_top1)
validacao_offline_limitacoes = [
    'Historico observado reflete a politica antiga de disparo, portanto ha vies de selecao.',
    'A cobertura do holdout por telefone escolhido e baixa; metricas offline sao proxy e nao prova causal.',
    'CPFs da dimensao e dos logs sao mascarados; a validacao offline usa desempenho agregado por telefone.',
]
if random_superou_champion:
    CHAMPION_MOTIVO += ' Random medio superou o champion em entrega top1; tratar como candidato para A/B, nao como vencedor offline.'
if cobertura_champion < 0.5:
    CHAMPION_MOTIVO += ' Cobertura de holdout inferior a 50%, reforcando necessidade de A/B.'

champion_status = 'candidato_para_ab_validacao_offline_fraca' if random_superou_champion or cobertura_champion < 0.5 else 'candidato_para_ab_validacao_offline_favoravel'
comparadores_bootstrap = [m for m in ['random_media_20_seeds', 'alfabetico', 'mais_recente', 'fonte_mais_recente', 'modelo', 'heuristica'] if m != CHAMPION_METODO]
bootstrap_comparacao = u.bootstrap_comparacao_metodos(metricas_top2_cpf_detalhe, CHAMPION_METODO, comparadores_bootstrap)
display(bootstrap_comparacao)

print('Champion:', CHAMPION_METODO)
print('Status:', champion_status)
print('Random superou champion:', random_superou_champion)
print(CHAMPION_MOTIVO)


## 6. Score operacional final e artefatos

A saida operacional continua gerando `telefone_1` e `telefone_2` por CPF. O campo `champion_metodo` e mantido para compatibilidade com o notebook 03, mas o status informa se o champion e apenas candidato por evidencia offline fraca.


In [ ]:
metricas_sistema_full, df_full_sistema = u.aprender_ranking_sistemas(df_disparo, df_aparicoes_fonte)
df_aparicoes_full_score = u.anexar_score_sistema(df_aparicoes_fonte, metricas_sistema_full)

df_eventos_full = u.montar_eventos(df_disparo, df_aparicoes_full_score, df_phone_meta, BEST_HALF_LIFE)
prior_ddd_full, baseline_ddd_full = u.calcular_prior_suavizado(df_eventos_full, 'telefone_ddd', 'y_entrega')
df_eventos_full_model = df_eventos_full.merge(prior_ddd_full, on='telefone_ddd', how='left').rename(columns={'score_prior': 'score_ddd'})
df_eventos_full_model['score_ddd'] = df_eventos_full_model['score_ddd'].fillna(baseline_ddd_full)

X_full = df_eventos_full_model[u.FEATURE_COLS].fillna(0)
y_full = df_eventos_full_model['y_entrega']
scaler_final = StandardScaler()
X_full_s = scaler_final.fit_transform(X_full)
modelo_final = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=u.SEED)
modelo_final.fit(X_full_s, y_full)

reference_time_final = df_disparo['envio_datahora'].max()
telefones_com_score = u.score_phones_at_reference(
    df_aparicoes_full_score, df_phone_meta, reference_time_final, BEST_HALF_LIFE,
    prior_ddd_full, baseline_ddd_full, scaler_final, modelo_final, u.FEATURE_COLS
)

if CHAMPION_METODO == 'modelo':
    sort_cols = ['score_modelo', 'score_heuristico', 'melhor_dias_atualizacao', 'telefone_numero']
    ascending = [False, False, True, True]
elif CHAMPION_METODO == 'heuristica':
    sort_cols = ['score_heuristico', 'melhor_dias_atualizacao', 'telefone_numero']
    ascending = [False, True, True]
elif CHAMPION_METODO == 'mais_recente':
    sort_cols = ['melhor_dias_atualizacao', 'score_heuristico', 'telefone_numero']
    ascending = [True, False, True]
elif CHAMPION_METODO == 'fonte_mais_recente':
    sort_cols = ['score_fonte_mais_recente', 'dias_fonte_mais_recente', 'score_heuristico', 'telefone_numero']
    ascending = [False, True, False, True]
else:
    sort_cols = ['telefone_numero']
    ascending = [True]

df_candidatos_final = df_phone_cpf.merge(telefones_com_score, on='telefone_numero', how='inner')
qtd_telefones_cpf = df_candidatos_final.groupby('cpf')['telefone_numero'].nunique()
cpfs_final_2mais = qtd_telefones_cpf[qtd_telefones_cpf >= 2].index
df_candidatos_final_2mais = df_candidatos_final[df_candidatos_final['cpf'].isin(cpfs_final_2mais)].copy()

melhores_por_cpf = u.selecionar_top2(df_candidatos_final_2mais, CHAMPION_METODO, sort_cols, ascending)
resultado_escolha = melhores_por_cpf[['cpf', 'telefone_numero', 'score_modelo', 'score_heuristico', 'rank']].pivot(
    index='cpf', columns='rank', values=['telefone_numero', 'score_modelo', 'score_heuristico']
)
resultado_escolha.columns = [f'{col[0]}_{col[1]}' for col in resultado_escolha.columns]
resultado_escolha = resultado_escolha.reset_index().rename(columns={
    'telefone_numero_1': 'telefone_1',
    'telefone_numero_2': 'telefone_2',
    'score_modelo_1': 'score_modelo_1',
    'score_modelo_2': 'score_modelo_2',
    'score_heuristico_1': 'score_heuristico_1',
    'score_heuristico_2': 'score_heuristico_2',
})

assert resultado_escolha['telefone_1'].notna().all()
assert resultado_escolha['telefone_2'].notna().all()

resumo_operacional = {
    'champion_metodo': CHAMPION_METODO,
    'champion_motivo': CHAMPION_MOTIVO,
    'champion_status': champion_status,
    'random_superou_champion': random_superou_champion,
    'total_cpfs_candidatos': int(qtd_telefones_cpf.shape[0]),
    'cpfs_com_1_telefone': int((qtd_telefones_cpf == 1).sum()),
    'cpfs_com_2_ou_mais_telefones': int((qtd_telefones_cpf >= 2).sum()),
    'cpfs_no_resultado_escolha': int(resultado_escolha['cpf'].nunique()),
    'telefone_2_nulos': int(resultado_escolha['telefone_2'].isna().sum()),
}

resumo_modelo_priorizacao = {
    'cutoff_tuning': cutoff_tuning,
    'cutoff_teste': cutoff_teste,
    'split_temporal': {k: splits[k] for k in ['frac_treino', 'frac_tuning', 'frac_teste']},
    'half_life_final': BEST_HALF_LIFE,
    'feature_cols': u.FEATURE_COLS,
    'metricas_validacao_modelo': metricas_validacao,
    'comparacao_offline': resumo_comparacao,
    'metricas_top2_cpf': metricas_top2_cpf,
    'bootstrap_comparacao': bootstrap_comparacao,
    'champion_metodo': CHAMPION_METODO,
    'champion_motivo': CHAMPION_MOTIVO,
    'champion_status': champion_status,
    'validacao_offline_limitacoes': validacao_offline_limitacoes,
    'random_superou_champion': random_superou_champion,
}

u.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
artefatos = {
    'resultado_escolha': resultado_escolha,
    'telefones_com_score': telefones_com_score,
    'metricas_sistema_full': metricas_sistema_full,
    'resumo_modelo_priorizacao': resumo_modelo_priorizacao,
    'resumo_operacional': resumo_operacional,
}
if CHAMPION_METODO == 'modelo':
    artefatos['modelo_logistico'] = modelo_final
    artefatos['scaler'] = scaler_final

for nome, obj in artefatos.items():
    caminho = u.PROCESSED_DIR / f'{nome}.pkl'
    with open(caminho, 'wb') as f:
        pickle.dump(obj, f)
    print(f'{nome} salvo em {caminho}')

print('\nResumo operacional:')
for chave, valor in resumo_operacional.items():
    print(f'{chave}: {valor}')

display(resultado_escolha.head(10))
